# Inverted cycle — physical metrics (magnetization, spin correlation, peak wavevector) | Kaggle

Pipeline: **parameters -> DDPM -> image -> Xception -> parameters**, evaluated with the
canonical three-metric physical set from `metrics.py`.

## Three-metric comparison
For each test-split parameter vector θ:
- **original** — the single real image, cropped to 39x39 before any physical metric.
- **generated** — the mean of the three metrics over `K_ENS` DDPM samples for that
  θ, each cropped 40->39 (top-left crop, exact — see `metrics.topleft_crop`) before
  any physical metric is computed.

Each of the three canonical metrics (`magnetization`, `spin_correlation`,
`peak_wave_vector`) is compared **original vs generated** via R² over the
evaluated test points.

> **TEST_FRACTION** (0-1) controls how much of the test split is evaluated: `1.0` = all, `0.1` = 10%.

> **Kaggle:** add the `dataset-spines-united-v2`, `weights-xception-model`,
> `weights-models` and `physicalmetrics` datasets from **+ Add Data**. Paths are
> auto-detected under `/kaggle/input`.

In [ ]:
import os, sys, math, time, glob, subprocess, gc
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

def _pip(pkg):
    try:
        __import__(pkg if pkg != "scikit-image" else "skimage")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_pip("scikit-image")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from skimage.metrics import structural_similarity as ssim_fn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TF {tf.__version__} | Torch {torch.__version__} | Device {DEVICE}")
for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception: pass

# ── Autodetección de rutas en Kaggle ──────────────────────────────────────────
def find_file(basename, roots=("/kaggle/input", ".")):
    for root in roots:
        hits = glob.glob(f"{root}/**/{basename}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"No se encontró {basename} bajo {roots}")

DATASET_PATH         = find_file("dataset_unificado_v2.npz")
XCEPTION_INV_WEIGHTS = find_file("modelo_xception_fulldatabaseV3100.h5")
DDPM_CHECKPOINT      = find_file("ddpm_spines_final_39crop.pt")
METRICS_PATH         = find_file("metrics.py")
for p in [DATASET_PATH, XCEPTION_INV_WEIGHTS, DDPM_CHECKPOINT, METRICS_PATH]:
    print("[OK]", p)

# -- Cargar metrics.py compartido (mismo patron que el resto de notebooks) ----
import importlib.util
spec = importlib.util.spec_from_file_location("metrics", METRICS_PATH)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics
from metrics import (MASK, topleft_crop, magnetization, spin_correlation,
                     peak_wave_vector, physical_metrics, physical_metrics_batch,
                     PHYSICAL_METRIC_NAMES, PHYSICAL_METRIC_LABELS)
print(f"metrics.py cargado desde: {METRICS_PATH} | MASK={MASK.shape} N_MASK={int(MASK.sum())}")


In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
PARAM_NAMES = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]
N_OUTPUTS   = 8
COND_DIM    = 8

# Splits (idénticos al entrenamiento original)
INV_SPLIT_SEED       = 42
INV_TEST_FRACTION    = 0.15
INV_VAL_FRACTION_REL = 0.1765
DDPM_SEED            = 42
DDPM_SUBSAMPLE_FRAC  = 1.0
DDPM_IMG_SIZE        = 40
TARGET_HW_INV        = (224, 224)
DDPM_FAST_STEPS      = 100
EPS_REL_FRAC         = 0.01

# ── Variable de control: fracción del SPLIT DE TEST a evaluar ─────────────────
#    1.0 = todo el test   ·   0.1 = 10% del test   ·   0.0 = nada
TEST_FRACTION = 0.10
BATCH_SEED    = 7

# Ensemble DDPM por punto (K muestras promediadas para la métrica "generada")
K_ENS         = 16          # muestras generadas por punto de parámetros
PTS_PER_BATCH = 16          # puntos por llamada al DDPM (P*K imágenes/llamada)

assert 0.0 <= TEST_FRACTION <= 1.0, "TEST_FRACTION debe estar en [0,1]"
print(f"TEST_FRACTION={TEST_FRACTION} | K_ENS={K_ENS} | DDPM_FAST_STEPS={DDPM_FAST_STEPS}")


In [ ]:
# ── Dataset + scalers ─────────────────────────────────────────────────────────
data   = np.load(DATASET_PATH, mmap_mode="r")
imgs   = data["img"]
params = np.asarray(data["params"])
labels = np.asarray(data["labels"]) if "labels" in data.files else None
N = imgs.shape[0]
print(f"Dataset: {N:,} | imgs {imgs.shape} | params {params.shape}")

all_idx = np.arange(N)
idx_train_pool, idx_test_inv, _, _ = train_test_split(
    all_idx, params, test_size=INV_TEST_FRACTION, random_state=INV_SPLIT_SEED)
idx_train_inv, idx_val_inv, _, _ = train_test_split(
    idx_train_pool, params[idx_train_pool],
    test_size=INV_VAL_FRACTION_REL, random_state=INV_SPLIT_SEED)
scaler_inv = MinMaxScaler().fit(params[idx_train_inv])

rng = np.random.RandomState(DDPM_SEED)
sub_idx = rng.choice(N, size=int(N*DDPM_SUBSAMPLE_FRAC), replace=False)
params_sub_raw = params[sub_idx]
imgs_sub = np.asarray(imgs[sub_idx]).astype(np.float32)
idx_all_sub = np.arange(len(sub_idx))
idx_tr_sub, idx_temp_sub = train_test_split(idx_all_sub, test_size=0.30, random_state=DDPM_SEED)
scaler_ddpm = MinMaxScaler().fit(params_sub_raw[idx_tr_sub])
imgs_train_np = imgs_sub[idx_tr_sub].astype(np.float32)
mn, mx = float(imgs_train_np.min()), float(imgs_train_np.max())
print(f"Test split: {len(idx_test_inv):,} | mn={mn:.4f} mx={mx:.4f}")
del imgs_sub, imgs_train_np; gc.collect()


In [ ]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dense,
                                     BatchNormalization, Dropout)
from tensorflow.keras.models import Model

def build_xception(n_outputs=8):
    inputs = Input(shape=(224, 224, 3), name="input_layer")
    base = Xception(weights=None, include_top=False, input_tensor=inputs)
    x = GlobalAveragePooling2D(name="global_average_pooling2d")(base.output)
    x = BatchNormalization(name="batch_normalization_4")(x)
    x = Dropout(0.4, name="dropout")(x)
    x = Dense(256, activation="relu", name="dense")(x)
    x = BatchNormalization(name="batch_normalization_5")(x)
    x = Dropout(0.3, name="dropout_1")(x)
    out = Dense(n_outputs, activation="linear", name="dense_1")(x)
    return Model(inputs, out, name="xception_inverso_V3100")

with tf.device("/cpu:0"):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_INV_WEIGHTS)
print(f"Xception cargado en CPU. params={xception_model.count_params():,}")


In [ ]:
T_STEPS, BETA_START, BETA_END = 1000, 1e-4, 0.02

class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="linear", device="cpu"):
        self.T = T
        if schedule == "linear":
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == "cosine":
            steps = T + 1; s = 0.008
            x = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x/T)+s)/(1+s)*math.pi*0.5)**2; ac = ac/ac[0]
            betas = (1 - ac[1:]/ac[:-1]).clamp(max=0.999)
        else: raise ValueError(schedule)
        alphas = 1.0 - betas; ac = torch.cumprod(alphas, dim=0)
        self.betas=betas; self.alphas=alphas; self.alphas_cumprod=ac
        self.sqrt_alphas_cumprod=ac.sqrt(); self.sqrt_one_minus_alphas_cumprod=(1.0-ac).sqrt()

def sinusoidal_embedding(t, dim):
    half = dim//2
    freqs = torch.exp(-math.log(10000)*torch.arange(half, device=t.device).float()/(half-1))
    args = t[:,None].float()*freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
    def forward(self, t, cond):
        return self.t_mlp(sinusoidal_embedding(t, self.t_mlp[0].in_features)) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1=nn.GroupNorm(groups,in_ch);  self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1)
        self.norm2=nn.GroupNorm(groups,out_ch); self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.emb_proj=nn.Linear(emb_dim,out_ch)
        self.dropout=nn.Dropout(dropout) if dropout>0 else nn.Identity()
        self.skip=nn.Conv2d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self, x, emb):
        h=F.silu(self.norm1(x)); h=self.conv1(h)
        h=h+self.emb_proj(F.silu(emb))[:,:,None,None]
        h=F.silu(self.norm2(h)); h=self.dropout(h); h=self.conv2(h)
        return h+self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm=nn.GroupNorm(groups,ch); self.qkv=nn.Conv2d(ch,ch*3,1); self.proj=nn.Conv2d(ch,ch,1)
    def forward(self, x):
        B,C,H,W=x.shape; h=self.norm(x)
        q,k,v=self.qkv(h).chunk(3,dim=1)
        q=q.reshape(B,C,-1); k=k.reshape(B,C,-1); v=v.reshape(B,C,-1)
        attn=torch.softmax(torch.bmm(q.transpose(1,2),k)/math.sqrt(C),dim=-1)
        return x+self.proj(torch.bmm(v,attn.transpose(1,2)).reshape(B,C,H,W))

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1,2,4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs=[base_ch*m for m in ch_mults]
        self.emb=TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in=nn.Conv2d(img_channels,chs[0],3,padding=1)
        self.down_blocks=nn.ModuleList(); self.down_samples=nn.ModuleList(); self.skip_channels=[]
        in_ch=chs[0]
        for i,out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([ResBlock(in_ch,out_ch,emb_dim,dropout=dropout),
                                                   ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch=out_ch
        self.mid_block1=ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.mid_attn=SelfAttention(chs[-1]); self.mid_block2=ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.up_blocks=nn.ModuleList(); self.up_samples=nn.ModuleList()
        for i,out_ch in enumerate(reversed(chs)):
            skip_ch=self.skip_channels[-(i+1)]
            self.up_blocks.append(nn.ModuleList([ResBlock(in_ch+skip_ch,out_ch,emb_dim,dropout=dropout),
                                                 ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch=out_ch
        self.norm_out=nn.GroupNorm(8,chs[0]); self.conv_out=nn.Conv2d(chs[0],img_channels,1)
    def forward(self, x, t, cond):
        emb=self.emb(t,cond); h=self.conv_in(x); skips=[]
        for (rb1,rb2),ds in zip(self.down_blocks,self.down_samples):
            h=rb1(h,emb); h=rb2(h,emb); skips.append(h); h=ds(h)
        h=self.mid_block1(h,emb); h=self.mid_attn(h); h=self.mid_block2(h,emb)
        for (rb1,rb2),us,sk in zip(self.up_blocks,self.up_samples,reversed(skips)):
            h=torch.cat([h,sk],dim=1); h=rb1(h,emb); h=rb2(h,emb); h=us(h)
        return self.conv_out(F.silu(self.norm_out(h)))

@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=DDPM_IMG_SIZE):
    model.eval(); B=cond.shape[0]
    x=torch.randn(B,1,img_size,img_size,device=cond.device)
    timesteps=list(range(0,scheduler.T,scheduler.T//n_steps))[::-1]
    for t_val in timesteps:
        t_t=torch.full((B,),t_val,device=cond.device,dtype=torch.long)
        eps=model(x,t_t,cond)
        sa=scheduler.sqrt_alphas_cumprod[t_val]; s1a=scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0=((x - s1a*eps)/sa).clamp(-1,1)
        if t_val>0:
            tp=max(t_val-scheduler.T//n_steps,0)
            x=scheduler.sqrt_alphas_cumprod[tp]*x0 + scheduler.sqrt_one_minus_alphas_cumprod[tp]*eps
        else:
            x=x0
    return x

print("Cargando DDPM...")
ckpt = torch.load(DDPM_CHECKPOINT, map_location=DEVICE, weights_only=False)
hp = ckpt["hyperparams"]; print("  hp:", hp)
ddpm_model = ConditionalUNet(img_channels=1, base_ch=hp["base_ch"], ch_mults=(1,2,4),
                             cond_dim=COND_DIM, emb_dim=hp["cond_emb_dim"], dropout=0.0).to(DEVICE)
state = ckpt["ema"] if ("ema" in ckpt and ckpt["ema"] is not None) else ckpt["model"]
ddpm_model.load_state_dict(state); ddpm_model.eval()
ddpm_scheduler = DDPMScheduler(T=T_STEPS, beta_start=BETA_START, beta_end=BETA_END,
                               schedule=hp["beta_schedule"], device=DEVICE)
print("  DDPM OK")


In [ ]:
def dataset_img_to_ddpm(img_39):
    x = np.asarray(img_39, dtype=np.float32)
    if x.ndim == 3: x = x[..., 0]
    x = np.pad(x, ((0,1),(0,1)), mode="reflect")          # 39->40
    x = (x - mn)/(mx - mn + 1e-8)
    return (x*2.0 - 1.0).astype(np.float32)

@torch.no_grad()
def generate_with_ddpm_batch(y_phys_batch, n_steps=DDPM_FAST_STEPS):
    cond = scaler_ddpm.transform(np.asarray(y_phys_batch, dtype=np.float32))
    cond_t = torch.tensor(cond, dtype=torch.float32, device=DEVICE)
    x = fast_sample(ddpm_model, cond_t, ddpm_scheduler, n_steps=n_steps, img_size=DDPM_IMG_SIZE)
    return x.detach().cpu().numpy()[:, 0]                  # (B,40,40)

XCEPTION_INFER_BATCH = 64
def infer_params_from_ddpm_batch(imgs_40, batch_size=XCEPTION_INFER_BATCH):
    out = []
    for s in range(0, len(imgs_40), batch_size):
        chunk = imgs_40[s:s+batch_size].astype(np.float32)[:, :39, :39]
        chunk = (chunk + 1.0)/2.0*(mx - mn) + mn
        chunk = chunk[..., None]
        chunk = tf.image.resize(chunk, TARGET_HW_INV)
        chunk = tf.image.grayscale_to_rgb(chunk)
        y_norm = xception_model.predict(chunk, verbose=0)
        out.append(scaler_inv.inverse_transform(y_norm))
    return np.concatenate(out, axis=0)
print("Inferencia configurada.")


In [ ]:
from IPython.display import display, HTML
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.grid":True,
                     "grid.color":"#cccccc","grid.linestyle":"--","grid.alpha":0.5,
                     "savefig.dpi":140,"savefig.bbox":"tight","font.family":"DejaVu Sans"})
COLOR_ORIG="#2c5f8d"; COLOR_GEN="#d77a3b"; COLOR_VAL="#2ca02c"

def _fmt(v, nd=4):
    if v is None: return "—"
    if isinstance(v,float) and (np.isnan(v) or np.isinf(v)): return "—"
    if isinstance(v,(int,np.integer)): return f"{int(v):d}"
    return f"{v:.{nd}f}"

def display_df_styled(df, title=None, num_cols=None, color_col=None, thresholds=(5,20), nd=4):
    cols=list(df.columns)
    if num_cols is None:
        num_cols=[c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    css="<style>.t{border-collapse:collapse;font-family:DejaVu Sans;font-size:12px}.t th{background:#2c5f8d;color:#fff;padding:6px 11px;border:1px solid #1d4666}.t td{padding:5px 11px;border:1px solid #dde2e6}.t tr:nth-child(even) td{background:#f5f7fa}.t td.num{text-align:right}.t td.good{color:#1b6f1b;font-weight:600}.t td.warn{color:#b45f06;font-weight:600}.t td.bad{color:#a30000;font-weight:600}.cap{color:#2c5f8d;font-weight:600;margin-top:8px}</style>"
    h=[css]
    if title: h.append(f'<p class="cap">{title}</p>')
    h.append('<table class="t"><thead><tr>'+''.join(f"<th>{c}</th>" for c in cols)+'</tr></thead><tbody>')
    for _,row in df.iterrows():
        h.append("<tr>")
        for c in cols:
            v=row[c]; cls="num" if c in num_cols else ""
            if color_col is not None and c==color_col:
                try:
                    av=abs(float(v))
                    cls += " good" if av<thresholds[0] else (" warn" if av<thresholds[1] else " bad")
                except Exception: pass
            h.append(f'<td class="{cls}">{_fmt(v,nd) if c in num_cols else v}</td>')
        h.append("</tr>")
    h.append("</tbody></table>"); display(HTML("".join(h)))
print("Estilo listo.")


---
## Physical metrics — canonical three-metric set (magnetization, spin correlation, peak wavevector)

For each test point θ we compare:
- **original** — the single real image (cropped to 39x39).
- **generated** — the mean of the three metrics over the `K_ENS` DDPM samples for
  that θ (each cropped to 39x39 before any metric is computed).

Comparison is by **R²(original vs generated)**, per metric, over the evaluated
test points.

In [ ]:
# ── Máscara de disco y métricas físicas — desde metrics.py (fuente de verdad) ─
# metrics.py construye MASK a 39x39 con radio 18.25 px; toda imagen debe cropearse
# a 39x39 con topleft_crop ANTES de evaluar cualquier métrica física.

# ── Imagen ────────────────────────────────────────────────────────────────────
def metric_mse(a,b):  return float(np.mean((a.astype(np.float64)-b.astype(np.float64))**2))
def metric_ssim(a,b): return float(ssim_fn(a,b,data_range=2.0))

print(f"Métricas físicas listas (metrics.py): {PHYSICAL_METRIC_NAMES} | "
      f"N_MASK={int(MASK.sum())} sitios en el disco 39x39.")


In [ ]:
# ── Selección del subconjunto de test según TEST_FRACTION ─────────────────────
n_eval = int(round(TEST_FRACTION*len(idx_test_inv)))
if n_eval < 1:
    raise ValueError(f"TEST_FRACTION={TEST_FRACTION} -> 0 muestras. Sube el valor.")
rng_eval = np.random.RandomState(BATCH_SEED)
if n_eval >= len(idx_test_inv):
    eval_idx = np.sort(np.asarray(idx_test_inv, dtype=np.int64))
else:
    eval_idx = np.sort(rng_eval.choice(idx_test_inv, size=n_eval, replace=False))
N_pts = len(eval_idx)
y_in_phys = params[eval_idx].astype(np.float32)
print(f"Evaluando {N_pts}/{len(idx_test_inv)} puntos de test "
      f"({100*N_pts/len(idx_test_inv):.1f}%) | imágenes a generar: {N_pts*K_ENS:,}")


---
## Evaluation: generate K images per point and compute the three canonical metrics
`orig_<metric>` (original, single image) vs `gen_<metric>` (generated, mean over
`K_ENS` DDPM samples). Both sides are cropped 40->39 (top-left, exact) before any
physical metric is computed.

In [ ]:
orig_M = {m: np.full(N_pts, np.nan) for m in PHYSICAL_METRIC_NAMES}   # original (1 imagen)
gen_M  = {m: np.full(N_pts, np.nan) for m in PHYSICAL_METRIC_NAMES}   # generado (media sobre K)
imgs_gen_rep = np.zeros((N_pts, DDPM_IMG_SIZE, DDPM_IMG_SIZE), dtype=np.float32)

t0 = time.time()
for ps in range(0, N_pts, PTS_PER_BATCH):
    pe = min(ps+PTS_PER_BATCH, N_pts)
    y_pts = y_in_phys[ps:pe]
    P = pe - ps
    y_rep = np.repeat(y_pts, K_ENS, axis=0)
    imgs_K = generate_with_ddpm_batch(y_rep).reshape(P, K_ENS, DDPM_IMG_SIZE, DDPM_IMG_SIZE)
    for j in range(P):
        k_pt = ps + j
        # original -> crop 40->39 (top-left, exacto) ANTES de cualquier metrica fisica
        img_o_39 = topleft_crop(dataset_img_to_ddpm(np.asarray(imgs[eval_idx[k_pt]])))
        mo = physical_metrics(img_o_39)
        for m in PHYSICAL_METRIC_NAMES: orig_M[m][k_pt] = mo[m]
        # generado: K muestras, cada una cropeada 40->39, media de las 3 metricas sobre K
        gk = imgs_K[j]
        imgs_gen_rep[k_pt] = gk[0]
        gk_39 = topleft_crop(gk)
        mg = physical_metrics_batch(gk_39)
        for m in PHYSICAL_METRIC_NAMES: gen_M[m][k_pt] = np.nanmean(mg[m])
    if (pe % (PTS_PER_BATCH*5) == 0) or pe == N_pts:
        el = time.time()-t0
        print(f"  {pe}/{N_pts} ({100*pe/N_pts:.0f}%)  {el/60:.1f} min  "
              f"ETA {el/max(pe,1)*(N_pts-pe)/60:.1f} min")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Listo en {(time.time()-t0)/60:.1f} min")
print("Inferencia Xception sobre representantes...")
y_pred_phys = infer_params_from_ddpm_batch(imgs_gen_rep)
print("  y_pred:", y_pred_phys.shape)


In [ ]:
# ── R² de parámetros (entrada vs re-estimado por Xception) ────────────────────
rows = []
for i, name in enumerate(PARAM_NAMES):
    yt = y_in_phys[:, i]; yp = y_pred_phys[:, i]
    rng_p = yt.max()-yt.min()
    rows.append({"param":name, "MAE":mean_absolute_error(yt,yp),
                 "RMSE":mean_squared_error(yt,yp)**0.5,
                 "R2": r2_score(yt,yp) if np.var(yt)>1e-12 else np.nan,
                 "nMAE_%": 100*mean_absolute_error(yt,yp)/rng_p if rng_p>0 else np.nan})
df_reg = pd.DataFrame(rows)
display_df_styled(df_reg, title=f"Parámetros: entrada vs re-estimado  (n={N_pts})",
                  num_cols=["MAE","RMSE","R2","nMAE_%"], color_col="nMAE_%",
                  thresholds=(5,15), nd=4)


---
## Physical comparison: `orig` vs `gen` (mean over K)
Magnetization M, spin correlation C_nn, and peak wavevector q_peak, compared
original vs generated via R².

In [ ]:
rows = []
for m in PHYSICAL_METRIC_NAMES:
    o = orig_M[m]; g = gen_M[m]
    v = np.isfinite(o) & np.isfinite(g)
    ov, gv = o[v], g[v]
    r2 = r2_score(ov, gv) if (len(ov)>=2 and np.var(ov)>1e-12) else np.nan
    rows.append({"métrica": m,
                 "mean(orig)": ov.mean() if len(ov) else np.nan,
                 "mean(gen)":  gv.mean() if len(gv) else np.nan,
                 "MAE": mean_absolute_error(ov,gv) if len(ov) else np.nan,
                 "R²(orig→gen)": r2, "n": int(v.sum())})
display_df_styled(pd.DataFrame(rows),
    title="orig vs gen (mean over K) — canonical three-metric physical comparison",
    num_cols=["mean(orig)","mean(gen)","MAE","R²(orig→gen)","n"], nd=4)

fig, axes = plt.subplots(1, len(PHYSICAL_METRIC_NAMES), figsize=(15, 4.7))
for ax, m in zip(axes, PHYSICAL_METRIC_NAMES):
    o = orig_M[m]; g = gen_M[m]
    v = np.isfinite(o) & np.isfinite(g); ov, gv = o[v], g[v]
    ax.scatter(ov, gv, s=14, alpha=0.5, c=COLOR_ORIG, edgecolors="white", linewidths=0.3)
    if len(ov) >= 2:
        lo, hi = min(ov.min(),gv.min()), max(ov.max(),gv.max()); pad=0.05*(hi-lo+1e-9)
        ax.plot([lo-pad,hi+pad],[lo-pad,hi+pad],"k--",lw=1)
        r2 = r2_score(ov,gv) if np.var(ov)>1e-12 else float("nan")
        ax.set_title(f"{PHYSICAL_METRIC_LABELS[m]}   $R^2$={r2:.3f}   n={len(ov)}")
        ax.set_xlim(lo-pad,hi+pad); ax.set_ylim(lo-pad,hi+pad)
    ax.set_xlabel("original"); ax.set_ylabel("generated (mean over K)")
fig.suptitle("Physical comparison: original vs generated (canonical three-metric set)")
plt.tight_layout(); plt.savefig("newmetrics_orig_vs_gen.png"); plt.show()


In [ ]:
# ── Guardar resultados ────────────────────────────────────────────────────────
np.savez_compressed("cycle_newmetrics_results.npz",
    eval_idx=eval_idx, y_in_phys=y_in_phys, y_pred_phys=y_pred_phys,
    TEST_FRACTION=TEST_FRACTION, K_ENS=K_ENS,
    **{f"orig__{m}": orig_M[m] for m in PHYSICAL_METRIC_NAMES},
    **{f"gen__{m}":  gen_M[m]  for m in PHYSICAL_METRIC_NAMES})
df_reg.to_csv("cycle_newmetrics_params.csv", index=False)
print("Guardado: cycle_newmetrics_results.npz, cycle_newmetrics_params.csv")
print(f"Resumen: TEST_FRACTION={TEST_FRACTION} | n={N_pts} | K_ENS={K_ENS}")
